# Confronto FEMBA (Foundation 2025) vs Corponi (WESAD)

## Obiettivo
Questo notebook implementa il confronto con **FEMBA**, un Foundation Model basato su Bidirectional Mamba.
Valuteremo se l'architettura FEMBA, pre-addestrata in modalità SSL sui 5 dataset, supera le performance della replica Corponi su WESAD.

In [ ]:
# 1. Setup Ambiente e Download FEMBA
import os, sys, shutil
!pip install -q --upgrade pip
!pip install -q packaging setuptools triton einops biosppy

print("Installazione causal-conv1d...")
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation

print("Installazione mamba-ssm...")
%env MAX_JOBS=1
!pip install -q mamba-ssm --no-build-isolation

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if os.path.exists(FEMBA_REPO): shutil.rmtree(FEMBA_REPO)
print("Cloning FEMBA Repository...")
!git clone --depth 1 https://github.com/pulp-bio/biofoundation.git {FEMBA_REPO}

import os
os._exit(0)

In [ ]:
# 2. Caricamento Modello FEMBA e Pesi (Hyperparameters Sync)
import os, sys, torch, torch.nn as nn, numpy as np, pandas as pd, glob, pickle
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if FEMBA_REPO not in sys.path: sys.path.append(FEMBA_REPO)
from models.FEMBA import FEMBA

device = 'cuda' if torch.cuda.is_available() else 'cpu'

weights_files = glob.glob('/kaggle/input/**/femba_encoder_pretrained.pt', recursive=True)
WEIGHTS_PATH = weights_files[0] if weights_files else None

# Usiamo ESATTAMENTE gli stessi parametri della Phase 1
model = FEMBA(
    seq_length=640,
    num_channels=6,
    num_classes=2,
    patch_size=(2, 16), # Sync con Phase 1
    stride=(2, 16),      # Sync con Phase 1
    embed_dim=128,
    num_blocks=4,
    classification_type="bc"
).to(device)

if WEIGHTS_PATH:
    print(f"✅ Caricamento pesi pre-addestrati da: {WEIGHTS_PATH}")
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)
else:
    print("⚠️ ATTENZIONE: Pesi non trovati.")


In [ ]:
# 3. Caricamento Dati WESAD
WESAD_DIR = glob.glob('/kaggle/input/**/WESAD', recursive=True)[0]
TARGET_LEN, LABEL_MAP = 640, {1: 0, 2: 1}

def segment_subject(data, label_signal):
    wrist = data['signal']['wrist']
    bvp, eda, temp, acc = wrist['BVP'].flatten(), wrist['EDA'].flatten(), wrist['TEMP'].flatten(), wrist['ACC']
    lbl = resample(label_signal.astype(float), len(bvp)).round().astype(int)
    segs, labs = [], []
    for start in range(0, len(bvp) - TARGET_LEN + 1, TARGET_LEN):
        maj = int(np.bincount(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]).argmax()) if len(lbl[start:start+TARGET_LEN][np.isin(lbl[start:start+TARGET_LEN], [1,2])]) > 0 else 0
        if maj not in LABEL_MAP: continue
        arr = np.stack([resample(acc[start//2:(start+TARGET_LEN)//2, i], TARGET_LEN) for i in range(3)] + [bvp[start:start+TARGET_LEN], resample(eda[start//16:(start+TARGET_LEN)//16], TARGET_LEN), resample(temp[start//16:(start+TARGET_LEN)//16], TARGET_LEN)])
        arr = (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-8)
        segs.append(arr); labs.append(LABEL_MAP[maj])
    return segs, labs

all_X, all_y, all_subs = [], [], []
for sdir in sorted(glob.glob(os.path.join(WESAD_DIR, 'S*'))):
    sid = os.path.basename(sdir)
    with open(os.path.join(sdir, f'{sid}.pkl'), 'rb') as f: data = pickle.load(f, encoding='latin1')
    segs, labs = segment_subject(data, data['label'])
    all_X.extend(segs); all_y.extend(labs); all_subs.extend([sid]*len(segs))


In [ ]:
# 4. Training Loop e Risultati
u_subs = np.unique(all_subs); np.random.seed(42); np.random.shuffle(u_subs)
tr_s, te_s = u_subs[:int(0.7*len(u_subs))], u_subs[int(0.7*len(u_subs)):]
tr_idx = np.isin(all_subs, tr_s)

class SigDataset(Dataset):
    def __init__(self, m): self.X, self.y = torch.tensor(np.array(all_X)[m]), torch.tensor(np.array(all_y)[m])
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

weights = 1. / np.array([len(np.where(np.array(all_y)[tr_idx] == t)[0]) for t in np.unique(np.array(all_y)[tr_idx])])
sampler = WeightedRandomSampler(torch.tensor([weights[int(t)] for t in np.array(all_y)[tr_idx]]), len(np.where(tr_idx)[0]))
train_loader = DataLoader(SigDataset(tr_idx), batch_size=32, sampler=sampler)
test_loader = DataLoader(SigDataset(np.isin(all_subs, te_s)), batch_size=32)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4); crit = nn.CrossEntropyLoss()
print("Starting FEMBA Fine-tuning (50 epochs)...")
for e in range(1, 51):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        mask = torch.zeros(xb.shape, dtype=torch.bool, device=device)
        pred, _ = model(xb.float().to(device), mask)
        loss = crit(pred, yb.to(device)); loss.backward(); opt.step()
    if e % 10 == 0: print(f"Epoch {e}/50 completata.")

model.eval(); p, t = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        mask = torch.zeros(xb.shape, dtype=torch.bool, device=device)
        pred, _ = model(xb.float().to(device), mask)
        p.extend(pred.argmax(1).cpu().numpy()); t.extend(yb.numpy())

acc = accuracy_score(t, p)
f1 = f1_score(t, p)
f1_macro = f1_score(t, p, average='macro')
print('\n--- CONFRONTO FINALE WESAD (FEMBA 2025) ---')
df = pd.DataFrame({
    'Modello': ['Corponi (Master)', 'FEMBA (SSL-Pretrained)'],
    'Accuracy': ['80.20%', f'{acc*100:.2f}%'],
    'F1-Score (Stress)': ['54.55%', f'{f1*100:.2f}%'],
    'Macro F1': ['70.95%', f'{f1_macro*100:.2f}%']
})
print(df.to_string(index=False))

# --- Aggiunta Matrice di Confusione ---
if 'confusion_matrix' not in locals() and 'confusion_matrix' not in globals():
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

cm = confusion_matrix(t, p)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baseline', 'Stress'])
disp.plot(cmap='Blues')
plt.title('Matrice di Confusione')
plt.show()
